In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import re
import subprocess
from pathlib import Path

QUEUE_REF = '100bb43a7c976a5d4ab751002441964fdb8b2724'  # immutable queue implementation SHA
DRIVE_ROOT = '/content/drive/MyDrive/t16_colab'
WORKER_READY_REF = None
assert re.fullmatch(r'[0-9a-f]{40}', QUEUE_REF) and set(QUEUE_REF) != {'0'}
bootstrap = Path('/content/colab-queue')
bootstrap.mkdir(parents=True, exist_ok=True)  # local runtime storage only
worker = bootstrap / 'worker.sh'
download = bootstrap / 'worker.sh.download'
url = f'https://raw.githubusercontent.com/josenunezwork/snake-dqn/{QUEUE_REF}/research/colab_queue/worker.sh'
subprocess.run(['curl', '-fsSL', '--retry', '3', '--max-time', '120', url, '-o', str(download)], check=True)
download.replace(worker)
WORKER_READY_REF = QUEUE_REF


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

assert WORKER_READY_REF == QUEUE_REF, 'Run the pinned fetch cell successfully first'
env = os.environ.copy()
env['QUEUE_REF'] = QUEUE_REF
env['DRIVE_ROOT'] = DRIVE_ROOT
env['COLAB_PYTHON'] = sys.executable
log_path = Path('/content/colab-queue/worker.log')
with log_path.open('ab', buffering=0) as log:
    proc = subprocess.Popen(
        ['nohup', 'bash', '/content/colab-queue/worker.sh'],
        cwd='/content/colab-queue', env=env, stdin=subprocess.DEVNULL, stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
    )
print(f'queue worker pid={proc.pid}; local log={log_path}; Drive root={env["DRIVE_ROOT"]}')
